# 01 — Exploratory Data Analysis
Understanding the credit card fraud dataset before modelling.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

df = pd.read_csv('../data/raw/creditcard.csv')
print(df.shape)
df.head()

In [ ]:
# Basic info
print(df.dtypes)
print('\nMissing values:')
print(df.isnull().sum())

In [ ]:
# Class imbalance — the core challenge
counts = df['Class'].value_counts()
fraud_rate = df['Class'].mean()
print(f'Legitimate: {counts[0]:,}')
print(f'Fraud:      {counts[1]:,}')
print(f'Fraud rate: {fraud_rate:.3%}')

fig, ax = plt.subplots()
ax.bar(['Legitimate', 'Fraud'], counts.values, color=['steelblue', 'crimson'])
ax.set_title('Class Distribution (extremely imbalanced)')
ax.set_ylabel('Count')
for i, v in enumerate(counts.values):
    ax.text(i, v + 200, f'{v:,}', ha='center', fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/class_distribution.png', dpi=150)
plt.show()

In [ ]:
# Transaction amount distribution by class
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, cls, label, colour in zip(axes, [0, 1], ['Legitimate', 'Fraud'], ['steelblue', 'crimson']):
    subset = df[df['Class'] == cls]['Amount']
    ax.hist(subset, bins=60, color=colour, alpha=0.8, edgecolor='white', linewidth=0.3)
    ax.set_title(f'{label} — Amount Distribution')
    ax.set_xlabel('Amount (£)')
    ax.set_ylabel('Count')
    ax.axvline(subset.median(), color='black', linestyle='--', label=f'Median: £{subset.median():.2f}')
    ax.legend()

plt.tight_layout()
plt.savefig('../reports/figures/amount_distribution.png', dpi=150)
plt.show()

In [ ]:
# Transaction frequency over time
df['hour'] = (df['Time'] // 3600) % 24

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, cls, label, colour in zip(axes, [0, 1], ['Legitimate', 'Fraud'], ['steelblue', 'crimson']):
    subset = df[df['Class'] == cls]['hour']
    ax.hist(subset, bins=24, range=(0, 24), color=colour, alpha=0.8, edgecolor='white')
    ax.set_title(f'{label} — Hour of Day')
    ax.set_xlabel('Hour')
    ax.set_xticks(range(0, 24, 2))

plt.tight_layout()
plt.savefig('../reports/figures/time_distribution.png', dpi=150)
plt.show()

In [ ]:
# Correlation heatmap (PCA features V1-V28)
v_cols = [c for c in df.columns if c.startswith('V')]
corr = df[v_cols + ['Amount', 'Class']].corr()

fig, ax = plt.subplots(figsize=(16, 12))
sns.heatmap(
    corr, cmap='RdBu_r', center=0, vmin=-1, vmax=1,
    square=True, linewidths=0.3, ax=ax
)
ax.set_title('Feature Correlation Matrix')
plt.tight_layout()
plt.savefig('../reports/figures/correlation_heatmap.png', dpi=150)
plt.show()

In [ ]:
# Top features correlated with fraud
fraud_corr = corr['Class'].drop('Class').abs().sort_values(ascending=False)
top_features = fraud_corr.head(15)

fig, ax = plt.subplots(figsize=(10, 6))
top_features.plot(kind='barh', ax=ax, color='steelblue')
ax.set_title('Top 15 Features by Correlation with Fraud')
ax.set_xlabel('|Correlation|')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('../reports/figures/top_feature_correlations.png', dpi=150)
plt.show()

print(top_features)

In [ ]:
# Distribution of top correlated features by class
top_v_features = [f for f in top_features.index if f.startswith('V')][:6]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, feat in zip(axes.flatten(), top_v_features):
    for cls, label, colour in [(0, 'Legit', 'steelblue'), (1, 'Fraud', 'crimson')]:
        data = df[df['Class'] == cls][feat]
        ax.hist(data, bins=60, alpha=0.6, color=colour, label=label, density=True)
    ax.set_title(feat)
    ax.legend(fontsize=8)

plt.suptitle('Feature Distributions by Class', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('../reports/figures/feature_distributions.png', dpi=150)
plt.show()

In [ ]:
# Summary statistics for fraud vs legit
df.groupby('Class')[['Amount'] + top_v_features[:5]].describe().T